# SafeGirl — V2 Intent Classifier Training

**5-fold StratifiedGroupKFold cross-validation, per the locked V2 methodology
(decision-log.md).**

This notebook imports and runs `train_distilbert.py` directly — it does not
duplicate the training logic inline, so the script (the actual source of
truth, version-controlled in git) and this notebook can never drift apart.

V1 (55 seeds, single split, 80% accuracy / macro-F1 0.7677) is a locked
historical baseline and is not touched by anything in this notebook.

# 1. Setup & Environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q \
    transformers==5.16.1 \
    datasets==4.0.0 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    accelerate==1.14.0

In [ ]:
import torch, transformers, datasets, sklearn, pandas, accelerate

print("=== SafeGirl V2 Training Environment ===")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Datasets:     {datasets.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"Pandas:       {pandas.__version__}")
print(f"Accelerate:   {accelerate.__version__}")
print()

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU detected. Stop before continuing with training.")

**Path configuration — everything under `MyDrive/SafeGirl/`, matching the
git repo structure.** `OUTPUT_DIR` points at Drive too, not local Colab
storage — checkpoints must survive a runtime disconnect.

In [ ]:
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SafeGirl")
NOTEBOOK_DIR = DRIVE_ROOT / "notebooks" / "classifier"

sys.path.insert(0, str(NOTEBOOK_DIR))

# Confirm train_distilbert.py is where we expect before importing it
script_path = NOTEBOOK_DIR / "train_distilbert.py"
if not script_path.exists():
    raise FileNotFoundError(
        f"train_distilbert.py not found at {script_path}. "
        "Upload it to this path in Drive before continuing."
    )
print(f"Found training script at: {script_path}")

In [ ]:
import train_distilbert as td

# Point the script's paths at Drive explicitly -- DATA_DIR alone is not
# enough; OUTPUT_DIR defaults to local Colab storage and must be
# overridden too, or checkpoints are lost on disconnect.
td.DATA_DIR = DRIVE_ROOT / "dataset"
td.OUTPUT_DIR = DRIVE_ROOT / "checkpoints"

print(f"DATA_DIR   = {td.DATA_DIR}")
print(f"OUTPUT_DIR = {td.OUTPUT_DIR}")

# 2. Dataset Loading & Fold Verification

No training happens in this section — this only loads and verifies the
data, using the exact same functions `train_distilbert.py` uses internally,
so what gets verified here is guaranteed to be what actually gets trained
on.

In [ ]:
seeds_path = td.DATA_DIR / "seeds" / "seeds.csv"
fold_path = td.DATA_DIR / "seeds" / "seed_split_assignment_v2.csv"
reviewed_dir = td.DATA_DIR / "reviewed"
test_dir = td.DATA_DIR / "test"

all_examples = td.build_all_examples(seeds_path, reviewed_dir)
print(f"Total examples (seeds + reviewed paraphrases): {len(all_examples)}")
print(f"Expected: 910 (182 seeds + 220 V1 paraphrases + 508 V2 paraphrases)")

assert len(all_examples) == 910, (
    f"Got {len(all_examples)}, expected 910 -- check dataset/reviewed/ "
    "actually contains all V1 AND V2 approved paraphrase files."
)

In [ ]:
fold_assignment = td.load_fold_assignment(fold_path)
folds = td.build_fold_datasets(all_examples, fold_assignment)

print(f"\nSum across folds: {sum(len(v) for v in folds.values())} (should equal {len(all_examples)})")

In [ ]:
test_examples = td.load_test_set(test_dir)
if test_examples is None:
    print("No independent test set yet (DR-04 still open). CV and the final")
    print("retrain can both proceed without it -- only the final one-time")
    print("test evaluation in Section 6 needs it.")
else:
    print(f"Independent test set found: {len(test_examples)} examples")

# 3. Model Configuration

Hyperparameters are defined once, in `train_distilbert.py` — displayed
here for the record, not redefined.

In [ ]:
print(f"Model:                    {td.MODEL_NAME}")
print(f"Classes:                  {len(td.CATEGORIES)} -> {td.CATEGORIES}")
print(f"Batch size:               {td.BATCH_SIZE}")
print(f"Learning rate:            {td.LEARNING_RATE}")
print(f"Max epochs:               {td.MAX_EPOCHS}")
print(f"Early stopping patience:  {td.EARLY_STOPPING_PATIENCE}")
print(f"Random seed:              {td.SEED}")
print(f"Number of folds:          {td.N_FOLDS}")
print(f"Model selection metric:   {td.MODEL_SELECTION_METRIC}  (macro-F1, locked at Step 12 checkpoint)")

**Load TensorBoard, pointed at Drive so logs survive a disconnect.**

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$td.OUTPUT_DIR"

# 4. Stage 1 — 5-Fold Cross-Validation

Five separate training runs, each with one fold held out as validation.
**Model selection and early stopping use macro-F1** — this is a real
correction relative to an earlier draft of this script, which used
weighted F1 by mistake (see decision-log.md). Watch the TensorBoard cell
above for live curves per fold.

In [ ]:
tokenizer = td.DistilBertTokenizerFast.from_pretrained(td.MODEL_NAME)

In [ ]:
fold_results = td.run_cross_validation(folds, tokenizer)

### 4.1 Aggregate CV Results

In [ ]:
cv_summary = td.aggregate_cv_results(fold_results)

**Record the mean ± SD and the aggregate confusion matrix here** (copy
the printed output above) before moving on — this is the number that
gets compared against V1, not any single fold's result.

# 5. Stage 2 — Final Model Retrain

A **fresh, sixth training run** on all 910 examples combined — not simply
the best-performing fold promoted to "final." Uses the average best-epoch
across the 5 folds as a fixed epoch count, since there is no held-out
validation set here to run early stopping against.

In [ ]:
final_trainer = td.train_final_model(folds, tokenizer, cv_summary["avg_best_epoch"])

# 6. Stage 3 — Independent Test Set Evaluation

Runs **exactly once**, only if `dataset/test/` actually contains a
genuinely independent expert-sourced test set (DR-04). Refuses to
substitute CV or validation results if it doesn't exist yet — do not
work around this by pointing it at validation data.

In [ ]:
test_metrics = td.evaluate_on_test_set(final_trainer, tokenizer, test_dir)

# 7. Experiment Log

Quick reference for comparing runs *within this notebook session* only.
**This is not the authoritative record** — after CV completes and again
after the final retrain, post a summary to `development-log.md` in the
actual repo. Kevin (or anyone else reviewing the project) checks the
git-tracked logs, not this notebook.

| Run | Stage | Macro-F1 (mean ± SD) | Accuracy | Notes |
|---|---|---|---|---|
| V2 | 5-fold CV | | | |
| V2 | Final retrain | (n/a — no held-out val) | | epochs used: |
| V2 | Independent test | | | pending DR-04 |
